# P-wave-earthquake-localisation-toy-box exemple notebook

In [1]:
import numpy as np

In [2]:
import generate_eq
import graphics
import commons

In [3]:
import Gauss_Newton
import Levenberg_Marquardt
import Uniq_searcher
import deepening_grid
import monte_carlo
import Evolutionary
import PSO_solver

In [4]:

# Deffine the stations, world box and V-P
#                        X       Y     Z    t
stations = np.array([[   0.0,  700.0,  0.0, 0.0],  # N°1
                     [ 700.0, 1000.0,  0.0, 0.0],  # N°2
                     [1000.0,  300.0,  0.0, 0.0],  # N°3
                     [ 500.0,  500.0, 10.0, 0.0],  # N°4
                     [ 400.0,    0.0,  0.0, 0.0]]) # N°5

box_gen = np.array([[    0.0,  1000.0],
                    [    0.0,  1000.0],
                    [-1000.0,     0.0]])

vp = 4000.0 # m/s

max_t = np.sum(np.max(box_gen**2, axis=1))**0.5 / vp

# Generate a random event without any noise
event, event_arr, stations, stations_true = generate_eq.generate_event(stations, box_gen, vp, 0.0)
print(f'Event before normalisation:\n\t{event}\n')

# Normalise the stations
stations_norm, event_norm = commons.centrering_arr(stations, event_arr)
event_norm_dict = {'X':float(event_norm[0]), 'Y':float(event_norm[1]),
                   'Z':float(event_norm[2]), 't':float(event_norm[3])}

print(f'Event after normalisation:\n\t{event_norm_dict}\n')

# Compute the normalised world box
limites = np.column_stack((np.min(stations_norm, axis=0), np.max(stations_norm, axis=0)))
limites[2, 0] = np.max(stations_norm[:, 2]) + box_gen[2, 0]
limites[3] = (-max_t, 0)

# Set the initialisation to be the same for all iterative solver
event_ini = np.random.rand(4)
event_ini = limites[:, 0] + event_ini * (limites[:, 1]-limites[:, 0])
event_ini[2] = min(event_ini[2], -1)
event_ini_dict = {'X':float(event_ini[0]), 'Y':float(event_ini[1]),
                  'Z':float(event_ini[2]), 't':float(event_ini[3])}

print(f'Initialisation:\n\t{event_ini_dict}\n')

Results = {}
Results['True event'] = {'X':float(event_norm[0]),
                         'Y':float(event_norm[1]),
                         'Z':float(event_norm[2]),
                         't':float(event_norm[3]),
                         'RMSE':0.0}


Event before normalisation:
	{'X': 183.04572186937452, 'Y': 805.5936998506729, 'Z': -350.315669904827, 't': 1785916525.4570262}

Event after normalisation:
	{'X': -316.9542781306255, 'Y': 305.5936998506729, 'Z': -360.315669904827, 't': -0.14223098754882812}

Initialisation:
	{'X': 373.10726232382046, 'Y': -346.3013596343667, 'Z': -20.35398899890947, 't': -0.02678214163655912}



In [5]:
event_best_GN, history_GN, cost_story_GN, misfit_fin_GN = Gauss_Newton.Gauss_Newton(
    stations=stations_norm,
    event_test=np.copy(event_ini),
    n_iteration=1_000,
    steps=np.array([1.0, 1.0, 1.0, 0.0005]),
    vp=vp)

print('Best fitt with:')
print(f'\t- RMSE = {misfit_fin_GN}')
print(f'\t- Event = {event_best_GN}')

Results['Gauss-Newton'] = {'X':event_best_GN['X'],
                           'Y':event_best_GN['Y'],
                           'Z':event_best_GN['Z'],
                           't':event_best_GN['t'],
                           'RMSE':misfit_fin_GN}

_ = graphics.history_2d(cost_story_GN, 'RMSE', log_y=True, filename=None)
_ = graphics.history_2d(history_GN[:, 3], 'Timing', filename=None)
_ = graphics.history_3d(history_GN[:, :3], stations_norm, filename=None)

Best fitt with:
	- RMSE = 5.471496669481552e-09
	- Event = {'X': -316.9542560888896, 'Y': 305.59355104174057, 'Z': -360.3152744345607, 't': -0.14223099768749176}


In [6]:
event_best_LM, history_LM, cost_story_LM, misfit_fin_LM = Levenberg_Marquardt.Levenberg_Marquardt(
    stations=stations_norm,
    event_test=np.copy(event_ini),
    n_iteration=1_000,
    vp=vp)

print('Best fitt with:')
print(f'\t- RMSE = {misfit_fin_LM}')
print(f'\t- Event = {event_best_LM}')

Results['Levenberg-Marquardt'] = {'X':event_best_LM['X'],
                                  'Y':event_best_LM['Y'],
                                  'Z':event_best_LM['Z'],
                                  't':event_best_LM['t'],
                                  'RMSE':misfit_fin_LM}

_ = graphics.history_2d(cost_story_LM, 'RMSE', log_y=True, filename=None)
_ = graphics.history_2d(history_LM[:, 3], 'Timing', filename=None)
_ = graphics.history_3d(history_LM[:, :3], stations_norm, filename=None)

Best fitt with:
	- RMSE = 5.471496663448801e-09
	- Event = {'X': -316.9542560888895, 'Y': 305.59355104174045, 'Z': -360.31527443456054, 't': -0.14223099768749173}


In [7]:
event_best_US, history_US, cost_story_US, misfit_fin_US = Uniq_searcher.gradient_descent(
    stations=stations_norm,
    event_test=np.copy(event_ini),
    n_iteration=200_000,
    steps=np.array([1.0, 1.0, 1.0, 0.0005]),
    lr_decay=0.05,
    patience=500,
    vp=4000.0)

print('Best fitt with:')
print(f'\t- RMSE = {misfit_fin_US}')
print(f'\t- Event = {event_best_US}')

Results['Unique searcher'] = {'X':event_best_US['X'],
                              'Y':event_best_US['Y'],
                              'Z':event_best_US['Z'],
                              't':event_best_US['t'],
                              'RMSE':misfit_fin_US}

_ = graphics.history_2d(cost_story_US, 'RMSE', log_y=True, filename=None)
_ = graphics.history_2d(history_US[:, 3], 'Timing', filename=None)
_ = graphics.history_3d(history_US[:, :3], stations_norm, filename=None)

Best fitt with:
	- RMSE = 6.267776573357929e-09
	- Event = {'X': -316.954216144546, 'Y': 305.5935228986021, 'Z': -360.31514335085564, 't': -0.14223097301311913}


In [8]:
event_best_DG, history_DG, cost_story_DG, misfit_fin_DG = deepening_grid.deepening_grid_search(
    stations=stations_norm,
    sampling_f=21,
    limites=np.copy(limites),
    depth=10001,
    halving_rate=0.20,
    vp=4000.0)

print('Best fitt with:')
print(f'\t- RMSE = {misfit_fin_DG}')
print(f'\t- Event = {event_best_DG}')

Results['Deepening grid'] = {'X':event_best_DG['X'],
                             'Y':event_best_DG['Y'],
                             'Z':event_best_DG['Z'],
                             't':event_best_DG['t'],
                             'RMSE':misfit_fin_DG}

_ = graphics.history_2d(cost_story_DG, 'RMSE', log_y=True, filename=None)
_ = graphics.history_2d(history_DG[:, 3], 'Timing', filename=None)
_ = graphics.history_3d(history_DG[:, :3], stations_norm, filename=None)

Best fitt with:
	- RMSE = 7.474370648005487e-09
	- Event = {'X': -316.9542236328125, 'Y': 305.5935363769531, 'Z': -360.3151550292969, 't': -0.14223097264766693}


In [9]:
event_best_DS, history_DS, cost_story_DS, misfit_fin_DS = deepening_grid.deepening_grid_sampling(
    stations=stations_norm,
    sampling_f=6,
    limites=np.copy(limites),
    depth=1001,
    halving_rate=0.20,
    n_samples=100,
    vp=4000.0)

print('Best fitt with:')
print(f'\t- RMSE = {misfit_fin_DS}')
print(f'\t- Event = {event_best_DS}')

Results['Deepening sampling'] = {'X':event_best_DS['X'],
                                 'Y':event_best_DS['Y'],
                                 'Z':event_best_DS['Z'],
                                 't':event_best_DS['t'],
                                 'RMSE':misfit_fin_DS}

_ = graphics.history_2d(cost_story_DS, 'RMSE', log_y=True, filename=None)
_ = graphics.history_2d(history_DS[:, 3], 'Timing', filename=None)
_ = graphics.history_3d(history_DS[:, :3], stations_norm, filename=None)

Best fitt with:
	- RMSE = 1.6514532684757425e-08
	- Event = {'X': -316.9542236328125, 'Y': 305.5935363769531, 'Z': -360.3155212402344, 't': -0.1422310322523117}


In [10]:
event_best_Mu, history_Mu, cost_story_Mu, misfit_fin_Mu = monte_carlo.monte_carlo(
    stations=stations_norm,
    limites=np.copy(limites),
    n_samples=10_000_000,
    sampling='uniform',
    vp=4000.0)

print('Best fitt with:')
print(f'\t- RMSE = {misfit_fin_Mu}')
print(f'\t- Event = {event_best_Mu}')

Results['Monte-Carlo uniform'] = {'X':event_best_Mu['X'],
                                  'Y':event_best_Mu['Y'],
                                  'Z':event_best_Mu['Z'],
                                  't':event_best_Mu['t'],
                                  'RMSE':misfit_fin_Mu}

_ = graphics.history_2d(cost_story_Mu, 'RMSE', log_y=True, filename=None)
_ = graphics.history_2d(history_Mu[:, 3], 'Timing', filename=None)
_ = graphics.history_3d(history_Mu[:, :3], stations_norm, filename=None)

Best fitt with:
	- RMSE = 0.0014728305395692587
	- Event = {'X': -341.0456237792969, 'Y': 321.04443359375, 'Z': -401.6304626464844, 't': -0.152128204703331}


In [11]:
event_best_Mg, history_Mg, cost_story_Mg, misfit_fin_Mg = monte_carlo.monte_carlo(
    stations=stations_norm,
    limites=np.copy(limites),
    n_samples=10_000_000,
    sampling='grid_rand',
    vp=4000.0)

print('Best fitt with:')
print(f'\t- RMSE = {misfit_fin_Mg}')
print(f'\t- Event = {event_best_Mg}')

Results['Monte-Carlo r grid'] = {'X':event_best_Mg['X'],
                                 'Y':event_best_Mg['Y'],
                                 'Z':event_best_Mg['Z'],
                                 't':event_best_Mg['t'],
                                 'RMSE':misfit_fin_Mg}

_ = graphics.history_2d(cost_story_Mg, 'RMSE', log_y=True, filename=None)
_ = graphics.history_2d(history_Mg[:, 3], 'Timing', filename=None)
_ = graphics.history_3d(history_Mg[:, :3], stations_norm, filename=None)

Best fitt with:
	- RMSE = 0.0008502171840518713
	- Event = {'X': -318.3343200683594, 'Y': 308.6895751953125, 'Z': -352.6768798828125, 't': -0.1410263031721115}


In [12]:
event_best_Mf, history_Mf, cost_story_Mf, misfit_fin_Mf = monte_carlo.monte_carlo(
    stations=stations_norm,
    limites=np.copy(limites),
    n_samples=10_000_000,
    sampling='fcc',
    vp=4000.0)

print('Best fitt with:')
print(f'\t- RMSE = {misfit_fin_Mf}')
print(f'\t- Event = {event_best_Mf}')

Results['Monte-Carlo fcc sampling'] = {'X':event_best_Mf['X'],
                                       'Y':event_best_Mf['Y'],
                                       'Z':event_best_Mf['Z'],
                                       't':event_best_Mf['t'],
                                       'RMSE':misfit_fin_Mf}

_ = graphics.history_2d(cost_story_Mf, 'RMSE', log_y=True, filename=None)
_ = graphics.history_2d(history_Mf[:, 3], 'Timing', filename=None)
_ = graphics.history_3d(history_Mf[:, :3], stations_norm, filename=None)

Best fitt with:
	- RMSE = 0.0011548034381121397
	- Event = {'X': -313.6695556640625, 'Y': 294.16510009765625, 'Z': -360.81854248046875, 't': -0.14074474573135376}


In [13]:
event_best_EV, history_EV, cost_story_EV, misfit_fin_EV = Evolutionary.evolution(
    pop_size=10_000,
    limites=np.copy(limites),
    n_iteration=1_000,
    stations=stations_norm,
    p_surv=0.10,
    patience=10,
    vp=4000.0)

print('Best fitt with:')
print(f'\t- RMSE = {misfit_fin_EV}')
print(f'\t- Event = {event_best_EV}')

Results['Evolutionary'] = {'X':event_best_EV['X'],
                           'Y':event_best_EV['Y'],
                           'Z':event_best_EV['Z'],
                           't':event_best_EV['t'],
                           'RMSE':misfit_fin_EV}

_ = graphics.history_2d(cost_story_EV, 'RMSE', log_y=True, filename=None)
_ = graphics.history_2d(history_EV[:, 3], 'Timing', filename=None)
_ = graphics.history_3d(history_EV[:, :3], stations_norm, filename=None)

Best fitt with:
	- RMSE = 9.428623962348867e-09
	- Event = {'X': -316.95424776586776, 'Y': 305.5935006553958, 'Z': -360.3151457706775, 't': -0.14223096912342653}


In [14]:
event_best_PSO, history_PSO, cost_story_PSO, misfit_fin_PSO = PSO_solver.evolution(
    stations=stations_norm,
    limites=np.copy(limites),
    n_iteration=1_000,
    pop_size=1_000,
    inertia=0.5,
    c_cogni=1.5,
    c_social=1.5,
    patience=10,
    vp=4000.0)

print('Best fitt with:')
print(f'\t- RMSE = {misfit_fin_PSO}')
print(f'\t- Event = {event_best_PSO}')

Results['PSO Solver'] = {'X':event_best_PSO['X'],
                         'Y':event_best_PSO['Y'],
                         'Z':event_best_PSO['Z'],
                         't':event_best_PSO['t'],
                         'RMSE':misfit_fin_PSO}

_ = graphics.history_2d(cost_story_PSO, 'RMSE', log_y=True, filename=None)
_ = graphics.history_2d(history_PSO[:, 3], 'Timing', filename=None)
_ = graphics.history_3d(history_PSO[:, :3], stations_norm, filename=None)

Best fitt with:
	- RMSE = 1.0690040642239915e-08
	- Event = {'X': -316.9543673289124, 'Y': 305.59357358013045, 'Z': -360.31558196569836, 't': -0.14223105035371852}


In [15]:
# Making some comparison
_ = graphics.solver_compare(Results,
                            None,
                            None,
                            title="Solver Comparison",
                            colorscale="Viridis",
                            filename=None,
                            height=1000)

In [18]:
_ = graphics.build_tables(Results, rounding=9)

+--------------------------+----------------+---------------+----------------+--------------+---------------+
|          Models          |       X        |       Y       |       Z        |      t       | RMSE (*10^-3) |
+--------------------------+----------------+---------------+----------------+--------------+---------------+
|        True event        | -316.954278131 | 305.593699851 | -360.315669905 | -0.142230988 |      0.0      |
+--------------------------+----------------+---------------+----------------+--------------+---------------+
|   Levenberg-Marquardt    | -316.954256089 | 305.593551042 | -360.315274435 | -0.142230998 |     5e-06     |
+--------------------------+----------------+---------------+----------------+--------------+---------------+
|       Gauss-Newton       | -316.954256089 | 305.593551042 | -360.315274435 | -0.142230998 |     5e-06     |
+--------------------------+----------------+---------------+----------------+--------------+---------------+
|     Uniq